# BRENDA — Enzyme Functional Data Ingestion and Analysis

**BRENDA** (BRaunschweig ENzyme DAtabase) is the world's most comprehensive collection of enzyme functional data. It is maintained at the Technische Universität Braunschweig and covers biochemical and molecular data for enzymes classified by their EC (Enzyme Commission) number.

Key data types stored in BRENDA:
| Data type | Description |
|---|---|
| **Kinetic parameters** | Km, Vmax, kcat, kcat/Km for substrate–enzyme pairs |
| **Substrates / Products** | Natural and synthetic substrates, reaction products |
| **Inhibitors / Activators** | Compounds that inhibit or activate enzyme activity |
| **Organism** | Source organism(s) for each reported parameter value |
| **Cofactors** | Metal ions, coenzymes, and prosthetic groups required |
| **References** | PubMed-linked literature supporting each data point |
| **Structure** | Links to PDB entries for 3D structural data |

Enzymes are identified by their **EC number** (e.g. `1.1.1.1` = alcohol dehydrogenase), a four-part hierarchical classification:
`[class].[subclass].[sub-subclass].[serial number]`

**API base URL:** `https://www.brenda-enzymes.org/api/v2/`  
**Documentation:** https://www.brenda-enzymes.org/rest-api.php  
**Reference:** Jeske et al. (2019), *Nucleic Acids Research*, BRENDA in 2019

# TODO

* [x] **Ingest data**
    * [x] Connect to BRENDA REST API and explore fields returned for EC 1.1.1.1 (alcohol dehydrogenase)
    * [x] Fetch enzyme entries for a set of well-known EC numbers; cache to `data/brenda_enzymes.json`
    * [x] Parse into a Polars DataFrame: ec_number, name, organism_count, substrate_count, km_values, references_count
    * [x] Fetch Km kinetic parameters for one enzyme; build a separate kinetics Polars DataFrame
    * [x] Print shape, dtypes, and head for both DataFrames
* [ ] **Explore and clean**
    * [ ] Summarize distributions of organism_count, substrate_count, references_count
    * [ ] Inspect Km value distributions; flag outliers and unit inconsistencies
    * [ ] Handle missing / null fields across both DataFrames
* [ ] **Analysis**
    * [ ] Compare kinetic parameters (Km, kcat) across organisms for a selected enzyme
    * [ ] Correlate substrate count with literature references count
    * [ ] Identify enzymes with extreme Km values and discuss biological implications
* [ ] **Visualization**
    * [ ] Bar chart of substrate and organism counts per EC number
    * [ ] Box/violin plot of Km distributions by organism or substrate
    * [ ] Scatter plot of Km vs. organism for a single enzyme (Bokeh or Seaborn)
* [ ] **Statistical analysis**
    * [ ] Discuss error propagation in experimentally reported Km values
    * [ ] Apply appropriate multiple-comparison corrections when testing across organisms
    * [ ] Primer on Michaelis–Menten kinetics and the mathematical derivation of Km

In [ ]:
import requests
import time
import json
from pathlib import Path

import polars as pl

## 1. Ingest Data

### 1.1 Connect to BRENDA API and Explore Fields for EC 1.1.1.1

In [ ]:
BRENDA_BASE = "https://www.brenda-enzymes.org/api/v2"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# EC numbers chosen to span all six primary enzyme classes:
#   1.1.1.1  Alcohol dehydrogenase       (Oxidoreductase)
#   2.7.1.1  Hexokinase                  (Transferase)
#   3.1.1.3  Triacylglycerol lipase      (Hydrolase)
#   4.2.1.1  Carbonic anhydrase          (Lyase)
EC_NUMBERS = ["1.1.1.1", "2.7.1.1", "3.1.1.3", "4.2.1.1"]
FOCUS_EC = "1.1.1.1"   # used for the detailed kinetics fetch in section 1.4


def brenda_get(ec_number: str, timeout: int = 30) -> requests.Response:
    """
    Send a GET request to the BRENDA REST API for one EC number.

    The public BRENDA REST API (v2) does not require authentication for
    basic enzyme summary endpoints, but rate-limits unauthenticated clients.
    A 1-second sleep is added after every call as a courtesy.

    Parameters
    ----------
    ec_number : str
        EC number in dotted notation, e.g. "1.1.1.1".
    timeout : int
        Request timeout in seconds.

    Returns
    -------
    requests.Response
        Raw response; caller handles JSON parsing and error checking.
    """
    url = f"{BRENDA_BASE}/enzyme/{ec_number}"
    resp = requests.get(url, timeout=timeout)
    time.sleep(1)   # polite delay — BRENDA asks for considerate use
    return resp


# ── Connectivity check: fetch EC 1.1.1.1 and print available top-level fields ─
resp = brenda_get(FOCUS_EC)

if resp.status_code == 200:
    data = resp.json()
    print(f"HTTP {resp.status_code}  — BRENDA API reachable")
    print(f"\nTop-level keys returned for EC {FOCUS_EC}:")
    for key, val in data.items():
        # Show type and a short preview for each field
        preview = str(val)[:80].replace("\n", " ")
        print(f"  {key:<30} {type(val).__name__:<10}  {preview}")
else:
    # BRENDA may require authentication or be temporarily unavailable.
    # We note this and will fall back to the EBI Rhea + UniProt APIs in section 1.2.
    print(f"HTTP {resp.status_code} — BRENDA API not available without auth (expected).")
    print("Will use EBI UniProt enzyme data as a public fallback.")
    data = None

### 1.2 Fetch Enzyme Entries and Cache to Disk

If the BRENDA REST API returns a non-200 status (it requires registration for some endpoints), we fall back to the **EBI UniProt API**, which exposes enzyme data — including EC classification, organism counts, and substrate annotations — through its open REST interface. Both sources are tried in order; whichever succeeds is stored in `data/brenda_enzymes.json`.

In [ ]:
UNIPROT_BASE = "https://rest.uniprot.org/uniprotkb"
ENZYMES_CACHE = DATA_DIR / "brenda_enzymes.json"


def fetch_brenda_entry(ec_number: str) -> dict | None:
    """
    Attempt to fetch an enzyme summary from the BRENDA REST API.

    Returns None if the API is unreachable or returns a non-200 status,
    signalling the caller to use the UniProt fallback instead.

    Parameters
    ----------
    ec_number : str
        EC number in dotted notation.

    Returns
    -------
    dict or None
        Parsed JSON from BRENDA, or None on failure.
    """
    try:
        resp = brenda_get(ec_number)
        if resp.status_code == 200:
            return resp.json()
    except requests.RequestException:
        pass
    return None


def fetch_uniprot_enzyme(ec_number: str) -> dict:
    """
    Fetch enzyme summary data from the EBI UniProt REST API for a given EC number.

    UniProt is used as a fallback when BRENDA requires authentication.
    The query retrieves reviewed (Swiss-Prot) entries annotated with the EC
    number, then aggregates organism count, substrate annotations, Km values
    mentioned in kinetics comments, and reference counts.

    Parameters
    ----------
    ec_number : str
        EC number in dotted notation, e.g. "1.1.1.1".

    Returns
    -------
    dict
        Aggregated summary with keys: ec_number, name, organisms, substrates,
        km_values, references_count, source.
    """
    # Query Swiss-Prot for reviewed entries with this EC annotation
    params = {
        "query": f"ec:{ec_number} AND reviewed:true",
        "fields": "id,protein_name,organism_name,cc_catalytic_activity,kinetics,lit_pubmed_id",
        "format": "json",
        "size": 100,   # cap at 100 entries for this exploration
    }
    resp = requests.get(f"{UNIPROT_BASE}/search", params=params, timeout=30)
    resp.raise_for_status()
    time.sleep(0.5)  # polite delay

    results = resp.json().get("results", [])
    if not results:
        return {
            "ec_number": ec_number, "name": None, "organisms": [],
            "substrates": [], "km_values": [], "references_count": 0,
            "source": "uniprot",
        }

    # Extract the recommended protein name from the first entry
    first = results[0]
    rec_names = (
        first.get("proteinDescription", {})
             .get("recommendedName", {})
    )
    name = (
        rec_names.get("fullName", {}).get("value")
        if rec_names else None
    )

    # Collect unique organism names across all returned entries
    organisms = list({
        r.get("organism", {}).get("scientificName", "")
        for r in results
        if r.get("organism", {}).get("scientificName")
    })

    # Parse substrates from catalytic activity annotations
    substrates = []
    for r in results:
        for comment in r.get("comments", []):
            if comment.get("commentType") == "CATALYTIC ACTIVITY":
                reaction = comment.get("reaction", {})
                # reaction.reactionCrossReferences gives ChEBI IDs;
                # reaction.name gives the full reaction string
                rxn_name = reaction.get("name", "")
                # Naive substrate extraction: take the left side of "=" in the reaction
                if "=" in rxn_name:
                    left = rxn_name.split("=")[0].strip()
                    # Split on " + " to get individual substrate names
                    for s in left.split(" + "):
                        s = s.strip()
                        if s and s not in substrates:
                            substrates.append(s)

    # Extract Km values mentioned in kinetics free-text comments
    km_values = []
    for r in results:
        for comment in r.get("comments", []):
            if comment.get("commentType") == "KINETICS":
                for text_block in comment.get("texts", []):
                    value_text = text_block.get("value", "")
                    # Look for patterns like "Km=0.15 mM" or "KM = 2.3 mM"
                    import re
                    for match in re.finditer(
                        r"[Kk][Mm]\s*=\s*([\d.]+)\s*(m?M|µM|uM)", value_text
                    ):
                        km_values.append(f"{match.group(1)} {match.group(2)}")

    # Count unique PubMed IDs across all entries as a references proxy
    pubmed_ids: set[str] = set()
    for r in results:
        for ref in r.get("references", []):
            for cross_ref in ref.get("citation", {}).get("citationCrossReferences", []):
                if cross_ref.get("database") == "PubMed":
                    pubmed_ids.add(cross_ref["id"])

    return {
        "ec_number":        ec_number,
        "name":             name,
        "organisms":        organisms,
        "substrates":       substrates,
        "km_values":        km_values,
        "references_count": len(pubmed_ids),
        "source":           "uniprot",
    }


def fetch_all_enzymes(ec_numbers: list[str], cache_path: Path = ENZYMES_CACHE) -> list[dict]:
    """
    Fetch enzyme summaries for a list of EC numbers.

    Tries the BRENDA REST API first; falls back to UniProt for any EC number
    where BRENDA is unavailable. Results are cached to disk.

    Parameters
    ----------
    ec_numbers : list[str]
        EC numbers to fetch.
    cache_path : Path
        File path for the JSON cache.

    Returns
    -------
    list[dict]
        One dict per EC number with summary fields.
    """
    if cache_path.exists():
        print(f"Loading from cache: {cache_path}")
        return json.loads(cache_path.read_text())

    entries = []
    for ec in ec_numbers:
        print(f"Fetching EC {ec} ...", end=" ")
        entry = fetch_brenda_entry(ec)   # try BRENDA first

        if entry is not None:
            # BRENDA succeeded — normalise to our common schema
            entry["source"] = "brenda"
            print(f"OK (BRENDA)  name={entry.get('name', entry.get('recommendedName', '?'))[:40]}")
        else:
            # Fall back to UniProt
            entry = fetch_uniprot_enzyme(ec)
            print(f"OK (UniProt) name={str(entry.get('name', '?'))[:40]}")

        entries.append(entry)

    cache_path.write_text(json.dumps(entries, indent=2))
    print(f"\nCached {len(entries)} entries → {cache_path}")
    return entries


enzymes_raw = fetch_all_enzymes(EC_NUMBERS)
print(f"\nTotal entries: {len(enzymes_raw)}")
print(f"Sources used : {set(e['source'] for e in enzymes_raw)}")

### 1.3 Parse into a Polars DataFrame

In [ ]:
def flatten_enzyme(entry: dict) -> dict:
    """
    Flatten one enzyme summary dict into a single DataFrame row.

    Nested list fields (organisms, substrates, km_values) are encoded as
    pipe-separated strings to fit in a single column; counts are computed
    from the list lengths before joining.

    Parameters
    ----------
    entry : dict
        Raw enzyme record as returned by fetch_all_enzymes.

    Returns
    -------
    dict
        Flat dict suitable for a Polars DataFrame row.
    """
    organisms  = entry.get("organisms") or []
    substrates = entry.get("substrates") or []
    km_values  = entry.get("km_values") or []

    return {
        "ec_number":        entry.get("ec_number"),
        "name":             entry.get("name"),
        "organism_count":   len(organisms),
        "substrate_count":  len(substrates),
        # Pipe-join preserves the full list in one string column
        "km_values":        " | ".join(km_values) if km_values else None,
        "references_count": entry.get("references_count", 0),
        "source":           entry.get("source"),
    }


rows = [flatten_enzyme(e) for e in enzymes_raw]   # one dict per EC number

enzymes = pl.DataFrame(rows).with_columns(
    pl.col("organism_count").cast(pl.Int32),
    pl.col("substrate_count").cast(pl.Int32),
    pl.col("references_count").cast(pl.Int32),
)

print("=== Enzymes DataFrame ===")
print(f"Shape  : {enzymes.shape}")
print(f"\nDtypes :")
print(enzymes.dtypes)
print(f"\nHead   :")
enzymes.head()

### 1.4 Fetch Km Kinetic Parameters for EC 1.1.1.1

In [ ]:
import re

KINETICS_CACHE = DATA_DIR / "brenda_kinetics.json"


def fetch_uniprot_kinetics(ec_number: str, max_entries: int = 200) -> list[dict]:
    """
    Fetch per-entry Km kinetic parameters from UniProt for a given EC number.

    Each UniProt entry can contain a KINETICS free-text comment with one or
    more Km values linked to specific substrates and organisms. This function
    parses those comments into structured records.

    Because Km values in UniProt are embedded in free text (e.g.
    "Km=0.15 mM for ethanol (in Homo sapiens)"), regex extraction is used.
    Values without an explicit substrate or organism are tagged as "unknown".

    Parameters
    ----------
    ec_number : str
        EC number in dotted notation, e.g. "1.1.1.1".
    max_entries : int
        Maximum number of UniProt entries to pull (each may contribute
        multiple Km rows).

    Returns
    -------
    list[dict]
        One dict per Km observation with keys: ec_number, uniprot_id,
        organism, substrate, km_value, km_unit.
    """
    params = {
        "query":  f"ec:{ec_number} AND reviewed:true",
        "fields": "id,organism_name,comments",
        "format": "json",
        "size":   min(max_entries, 500),
    }
    resp = requests.get(f"{UNIPROT_BASE}/search", params=params, timeout=30)
    resp.raise_for_status()
    time.sleep(0.5)

    results = resp.json().get("results", [])
    records = []

    for entry in results:
        uniprot_id = entry.get("primaryAccession", "")
        organism   = entry.get("organism", {}).get("scientificName", "unknown")

        for comment in entry.get("comments", []):
            if comment.get("commentType") != "KINETICS":
                continue

            for text_block in comment.get("texts", []):
                text = text_block.get("value", "")

                # Pattern: "Km=<value> <unit> for <substrate>"
                # Also handles "KM = <value> <unit> for <substrate>" variants
                for match in re.finditer(
                    r"[Kk][Mm]\s*=\s*([\d.]+)\s*(m?M|µM|uM|nM)"
                    r"(?:\s+for\s+([^(;.,]+?))?(?:\s*[;.,]|$)",
                    text,
                ):
                    km_val_str  = match.group(1)
                    km_unit     = match.group(2)
                    substrate   = (match.group(3) or "unknown").strip()

                    try:
                        km_value = float(km_val_str)
                    except ValueError:
                        continue   # skip malformed values

                    records.append({
                        "ec_number":  ec_number,
                        "uniprot_id": uniprot_id,
                        "organism":   organism,
                        "substrate":  substrate,
                        "km_value":   km_value,
                        "km_unit":    km_unit,
                    })

    return records


def fetch_kinetics(ec_number: str, cache_path: Path = KINETICS_CACHE) -> list[dict]:
    """
    Fetch and cache Km kinetics records for one EC number.

    Parameters
    ----------
    ec_number : str
        EC number in dotted notation.
    cache_path : Path
        JSON cache path.

    Returns
    -------
    list[dict]
        Km observation records.
    """
    if cache_path.exists():
        print(f"Loading kinetics from cache: {cache_path}")
        return json.loads(cache_path.read_text())

    print(f"Fetching Km records for EC {ec_number} from UniProt ...")
    records = fetch_uniprot_kinetics(ec_number)
    cache_path.write_text(json.dumps(records, indent=2))
    print(f"Cached {len(records)} Km records → {cache_path}")
    return records


kinetics_raw = fetch_kinetics(FOCUS_EC)
print(f"\nRaw Km records for EC {FOCUS_EC}: {len(kinetics_raw)}")

### 1.5 Build the Kinetics DataFrame and Summarise Both DataFrames

In [ ]:
# Build the kinetics DataFrame; cast to precise numeric types
if kinetics_raw:
    kinetics = pl.DataFrame(kinetics_raw).with_columns(
        pl.col("km_value").cast(pl.Float64),   # Km in original reported unit
    )
else:
    # Empty fallback schema so downstream cells don't break
    kinetics = pl.DataFrame(schema={
        "ec_number":  pl.Utf8,
        "uniprot_id": pl.Utf8,
        "organism":   pl.Utf8,
        "substrate":  pl.Utf8,
        "km_value":   pl.Float64,
        "km_unit":    pl.Utf8,
    })

# ── Enzymes DataFrame summary ─────────────────────────────────────────────────
print("=" * 60)
print("ENZYMES DATAFRAME")
print("=" * 60)
print(f"Shape   : {enzymes.shape}")
print(f"\nDtypes  :")
for name, dtype in zip(enzymes.columns, enzymes.dtypes):
    print(f"  {name:<20} {dtype}")
print(f"\nHead    :")
print(enzymes)

# ── Kinetics DataFrame summary ────────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"KINETICS DATAFRAME  (EC {FOCUS_EC})")
print("=" * 60)
print(f"Shape   : {kinetics.shape}")
print(f"\nDtypes  :")
for name, dtype in zip(kinetics.columns, kinetics.dtypes):
    print(f"  {name:<20} {dtype}")
print(f"\nHead    :")
print(kinetics.head(10))